# Local Knowledge Graph — PMAnalyze (offline 6.4.10)

Этот notebook **повторяет интерфейс панели Knowledge Graph** из дашборда, но локально и офлайн.

Что внутри (как в веб-панели):
- поле ввода запроса, кнопка поиска, ползунок **глубины связей**, флажок **LLM-саммари**;
- **панель визуала** графа (vis-network): узлы knowledge/wiki, связи relation/in_wiki, хабы, подсветка;
- **карточки** узлов по клику (заголовок, meta, тело, источник);
- **гибридный поиск** BM25 (SQLite FTS5) + вектор (cosine), слияние RRF — как `/api/kg/search`.

Источник данных — НЕ PostgreSQL. Берём **уже выгруженный** архив с интерфейса (кнопка «выгрузить» на панели графа):
- **Вариант A:** `pm_knowledge_local.sqlite`
- **Вариант B:** набор `*.csv` (knowledge, knowledge_relations, ...)

Для запуска нужны две модели:
- **модель векторизации** (SentenceTransformer) — размерность **детектится автоматически**;
- **модель саммари** (опционально, OpenAI-совместимый endpoint).


## 1) Установка зависимостей

Если окружение чистое — раскомментируйте `%pip install`.

In [ ]:
# %pip install -r requirements.txt
import importlib, sys
_need = ["pandas", "numpy", "sentence_transformers", "pyvis"]
_miss = [m for m in _need if importlib.util.find_spec(m) is None]
print("Отсутствуют:", _miss or "нет — всё на месте")
if _miss:
    print("Выполните:  %pip install -r requirements.txt")


## 2) Импорты и конфиг

Выбор источника (`SOURCE_MODE`) и путей. Всё через переменные окружения с дефолтами.

In [ ]:
import os, io, json, math, time, sqlite3, pathlib, zipfile, re
from typing import List, Optional
import numpy as np
import pandas as pd

BASE_DIR = pathlib.Path.cwd()
DATA_DIR = pathlib.Path(os.getenv("PM_DATA_DIR", str(BASE_DIR)))

# ── Источник данных: "sqlite" (вариант A) или "csv" (вариант B) ─────────────
SOURCE_MODE = os.getenv("PM_SOURCE_MODE", "sqlite").strip().lower()   # sqlite | csv
SQLITE_PATH = pathlib.Path(os.getenv("PM_SQLITE", str(DATA_DIR / "pm_knowledge_local.sqlite")))
CSV_DIR     = pathlib.Path(os.getenv("PM_CSV_DIR", str(DATA_DIR)))

# Рабочая локальная БД (создаётся из CSV при SOURCE_MODE=csv, иначе = SQLITE_PATH)
WORK_DB = SQLITE_PATH if SOURCE_MODE == "sqlite" else (BASE_DIR / "artifacts" / "pm_local_from_csv.sqlite")
WORK_DB.parent.mkdir(parents=True, exist_ok=True)

# ── Модель векторизации (размерность детектится автоматически) ──────────────
EMBED_MODEL_NAME = os.getenv("PM_LOCAL_EMBED_MODEL",
                             "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
EMBED_BATCH = int(os.getenv("PM_EMBED_BATCH", "64"))

# ── Бэкенд векторизации: "local" (SentenceTransformer, офлайн) или
#    "openai" (OpenAI-совместимый /embeddings как в проде PMAnalyze) ─────────
EMBED_BACKEND = os.getenv("PM_EMBED_BACKEND", "local").strip().lower()  # local | openai
EMBED_API = {
    "base_url": os.getenv("PM_EMBED_BASE", ""),   # напр. https://foundation-models.api.cloud.ru/v1
    "api_key":  os.getenv("PM_EMBED_KEY", ""),
    "model":    os.getenv("PM_EMBED_MODEL", "Qwen/Qwen3-VL-Embedding-8B"),
    "dim":      int(os.getenv("PM_EMBED_DIM", "0") or 0),  # 0 = автодетект по первому ответу
    "timeout":  float(os.getenv("PM_EMBED_TIMEOUT", "60")),
}

# ── Модель саммари (опционально, OpenAI-совместимый API) ────────────────────
LLM_CFG = {
    "base_url": os.getenv("PM_LLM_BASE_URL", ""),      # напр. https://foundation-models.api.cloud.ru/v1
    "api_key":  os.getenv("PM_LLM_API_KEY", ""),
    "model":    os.getenv("PM_LLM_MODEL", "anthropic/claude-haiku-4.5"),
}

# ── Параметры графа/поиска (совпадают с продовой панелью) ────────────────────
KG = dict(
    DEPTH_DEFAULT=1,       # глубина связей по умолчанию (как в UI)
    RRF_K=60,             # константа слияния RRF
    MAX_CTX=100,          # сколько узлов уходит в LLM
    LIMIT=120,            # сколько кандидатов ищем
    VEC_MIN_SIM=0.50,     # порог косинуса
    HUB_DEGREE=6,         # степень для «хаба»
)

print("SOURCE_MODE :", SOURCE_MODE)
print("WORK_DB     :", WORK_DB)
print("EMBED_BACKEND:", EMBED_BACKEND)
print("EMBED_MODEL :", EMBED_MODEL_NAME if EMBED_BACKEND=="local" else EMBED_API["model"])
print("LLM model   :", LLM_CFG["model"], "(", "on" if LLM_CFG["api_key"] else "off", ")")


## 3) Загрузка данных: вариант A (SQLite) / вариант B (CSV)

Оба варианта приводятся к одной локальной SQLite `WORK_DB` с таблицами:
`knowledge`, `knowledge_relations` (+ по возможности `wiki_pages`).
Схема гибкая — читаем те колонки, что есть, без жёсткой привязки.

In [ ]:
CSV_TABLES = ["knowledge", "metadata_knowledge", "knowledge_review",
              "knowledge_tags", "knowledge_relations", "wiki_pages"]

def _read_csv_dir(csv_dir: pathlib.Path) -> dict:
    out = {}
    for t in CSV_TABLES:
        p = csv_dir / f"{t}.csv"
        if p.exists():
            out[t] = pd.read_csv(p, dtype=str, keep_default_na=False)
    return out

def build_work_db():
    if SOURCE_MODE == "sqlite":
        assert SQLITE_PATH.exists(), f"Не найден sqlite: {SQLITE_PATH}"
        print("Вариант A: используем готовый sqlite ->", SQLITE_PATH)
        return
    # Вариант B: собираем sqlite из CSV
    frames = _read_csv_dir(CSV_DIR)
    assert frames, f"Не найдено ни одного CSV в {CSV_DIR}"
    if WORK_DB.exists():
        WORK_DB.unlink()
    with sqlite3.connect(WORK_DB) as sq:
        for t, df in frames.items():
            df.to_sql(t, sq, if_exists="replace", index=False)
    print("Вариант B: собрали sqlite из CSV ->", WORK_DB, "| таблицы:", list(frames.keys()))

def table_exists(db: pathlib.Path, name: str) -> bool:
    with sqlite3.connect(db) as sq:
        r = sq.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (name,)).fetchone()
    return bool(r)

def read_table(db: pathlib.Path, name: str) -> pd.DataFrame:
    with sqlite3.connect(db) as sq:
        return pd.read_sql_query(f'SELECT * FROM "{name}"', sq)

build_work_db()
print("knowledge?", table_exists(WORK_DB, "knowledge"),
      "| relations?", table_exists(WORK_DB, "knowledge_relations"),
      "| wiki?", table_exists(WORK_DB, "wiki_pages"))
kn = read_table(WORK_DB, "knowledge")
print("knowledge rows:", len(kn), "| cols:", list(kn.columns)[:12])
kn.head(3)


## 4) Модель векторизации (два бэкенда) + автодетект размерности

Переключается переменной `PM_EMBED_BACKEND`:

- **`local`** (по умолчанию) — `SentenceTransformer`, работает офлайн без ключей.
- **`openai`** — OpenAI-совместимый `POST {PM_EMBED_BASE}/embeddings` (те же вектора, что в проде PMAnalyze; нужны `PM_EMBED_KEY`, `PM_EMBED_MODEL`).

`VECTOR_DIM` в обоих режимах определяется по факту первого прогона — никаких хардкодов 384/768/4096. Параметры см. в `.env.example`.


In [ ]:
# Два режима векторизации, переключаются EMBED_BACKEND (ячейка конфига).
# local  — SentenceTransformer, офлайн, без ключей.
# openai — OpenAI-совместимый POST {base_url}/embeddings (те же вектора, что в проде).
import urllib.request, urllib.error

_model = None
VECTOR_DIM = None


def _l2_normalize(mat: np.ndarray) -> np.ndarray:
    mat = np.asarray(mat, dtype=np.float32)
    n = np.linalg.norm(mat, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return mat / n


# ────────────────── backend: local ──────────────────
def _get_local_model():
    global _model, VECTOR_DIM
    if _model is None:
        from sentence_transformers import SentenceTransformer
        print("Загружаем локальную модель:", EMBED_MODEL_NAME, "…")
        _model = SentenceTransformer(EMBED_MODEL_NAME)
        probe = _model.encode(["dimension probe"], normalize_embeddings=True)
        VECTOR_DIM = int(np.asarray(probe).shape[1])
        print("Автодетект VECTOR_DIM (local) =", VECTOR_DIM)
    return _model


def _embed_local(texts: List[str]) -> np.ndarray:
    m = _get_local_model()
    vecs = m.encode(list(texts), batch_size=EMBED_BATCH,
                    normalize_embeddings=True, show_progress_bar=False)
    return np.asarray(vecs, dtype=np.float32)


# ────────────────── backend: openai ─────────────────
def _embed_openai_batch(texts: List[str]) -> np.ndarray:
    base = EMBED_API["base_url"].rstrip("/")
    if not base or not EMBED_API["api_key"]:
        raise RuntimeError("EMBED_BACKEND=openai, но не заданы PM_EMBED_BASE / PM_EMBED_KEY")
    payload = json.dumps({"model": EMBED_API["model"], "input": list(texts)}).encode("utf-8")
    req = urllib.request.Request(
        base + "/embeddings",
        data=payload,
        headers={"Authorization": "Bearer " + EMBED_API["api_key"],
                 "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=EMBED_API["timeout"]) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    rows = sorted(data["data"], key=lambda d: d.get("index", 0))
    return np.asarray([r["embedding"] for r in rows], dtype=np.float32)


def _embed_openai(texts: List[str]) -> np.ndarray:
    global VECTOR_DIM
    out = []
    step = max(1, EMBED_BATCH)
    for i in range(0, len(texts), step):
        out.append(_embed_openai_batch(texts[i:i + step]))
    mat = np.vstack(out) if out else np.zeros((0, VECTOR_DIM or 1), dtype=np.float32)
    if VECTOR_DIM is None and mat.shape[0]:
        VECTOR_DIM = int(mat.shape[1])
        want = EMBED_API["dim"]
        if want and want != VECTOR_DIM:
            print(f"[warn] PM_EMBED_DIM={want}, но API вернул {VECTOR_DIM} — использую фактическую")
        print("Автодетект VECTOR_DIM (openai) =", VECTOR_DIM)
    return _l2_normalize(mat)


# ────────────────── единая точка входа ──────────────
def embed(texts: List[str]) -> np.ndarray:
    if EMBED_BACKEND == "openai":
        return _embed_openai(list(texts))
    return _embed_local(list(texts))


def warmup_embeddings():
    """Прогрев + автодетект размерности выбранного бэкенда."""
    v = embed(["dimension probe"])
    return int(v.shape[1]) if v.shape[0] else None


_dim = warmup_embeddings()
print("Готово. Бэкенд:", EMBED_BACKEND, "| Размерность вектора:", VECTOR_DIM)


## 5) Индексация: тексты knowledge/wiki + FTS5 (BM25) + матрица векторов

Строим то же «поисковое пространство», что и на бэке:
- FTS5-таблица для BM25 (аналог `tsvector` + `ts_rank_cd`);
- матрица нормализованных векторов для cosine (аналог pgvector `<=>`).

In [ ]:
def _first_col(df, *cands):
    for c in cands:
        if c in df.columns:
            return c
    return None

def load_nodes() -> pd.DataFrame:
    kn = read_table(WORK_DB, "knowledge")
    idc  = _first_col(kn, "id")
    txtc = _first_col(kn, "text_knowledge", "text", "content")
    impc = _first_col(kn, "importance")
    stc  = _first_col(kn, "status")
    rows = []
    for _, r in kn.iterrows():
        rows.append({
            "node_id": f"k{r[idc]}",
            "kind": "knowledge",
            "text": str(r.get(txtc, "") or ""),
            "importance": float(r.get(impc, 0) or 0) if impc else 0.0,
            "status": str(r.get(stc, "") or "") if stc else "",
        })
    if table_exists(WORK_DB, "wiki_pages"):
        wk = read_table(WORK_DB, "wiki_pages")
        widc = _first_col(wk, "id")
        titc = _first_col(wk, "title")
        mdc  = _first_col(wk, "content_md", "content")
        for _, r in wk.iterrows():
            title = str(r.get(titc, "") or "")
            body  = str(r.get(mdc, "") or "")
            rows.append({
                "node_id": f"w{r[widc]}", "kind": "wiki",
                "text": (title + "\n\n" + body).strip(),
                "importance": 0.5, "status": "",
                "title": title,
            })
    return pd.DataFrame(rows)

NODES = load_nodes()
NODES["label"] = NODES["text"].str.slice(0, 72) + NODES["text"].str.len().gt(72).map({True:"…", False:""})
print("Узлов:", len(NODES), "| knowledge:", (NODES.kind=="knowledge").sum(), "| wiki:", (NODES.kind=="wiki").sum())

# --- FTS5 индекс (BM25) ---
FTS_DB = BASE_DIR / "artifacts" / "kg_fts.sqlite"
FTS_DB.parent.mkdir(parents=True, exist_ok=True)
if FTS_DB.exists(): FTS_DB.unlink()
_fts = sqlite3.connect(FTS_DB)
_fts.execute("CREATE VIRTUAL TABLE nodes_fts USING fts5(node_id UNINDEXED, kind UNINDEXED, body)")
_fts.executemany("INSERT INTO nodes_fts(node_id, kind, body) VALUES (?,?,?)",
                 [(r.node_id, r.kind, r.text) for r in NODES.itertuples()])
_fts.commit()
print("FTS5 построен:", FTS_DB)

# --- Матрица векторов ---
EMB = embed(NODES["text"].fillna("").tolist())
print("Матрица векторов:", EMB.shape, "(dim =", VECTOR_DIM, ")")


## 6) Гибридный поиск BM25 + вектор (RRF) — как `/api/kg/search`

Возвращает ранжированный список узлов. RRF-слияние двух ранжировок,
константа `rrf_k` совпадает с продом (60).

In [ ]:
def _fts_query(q: str):
    # экранируем под FTS5 MATCH: разбиваем на токены, оборачиваем в кавычки
    toks = re.findall(r"[\w\-]+", q, flags=re.UNICODE)
    if not toks: return None
    return " OR ".join(f'"{t}"' for t in toks)

def bm25_search(q: str, limit: int) -> List[str]:
    mq = _fts_query(q)
    if not mq: return []
    cur = _fts.execute(
        "SELECT node_id, bm25(nodes_fts) AS s FROM nodes_fts WHERE nodes_fts MATCH ? ORDER BY s LIMIT ?",
        (mq, limit))
    return [row[0] for row in cur.fetchall()]   # bm25(): меньше = лучше, уже отсортировано

def vec_search(q: str, limit: int, min_sim: float) -> List[str]:
    qv = embed([q])[0]
    sims = EMB @ qv                              # cosine (векторы нормированы)
    order = np.argsort(-sims)
    out = []
    for i in order:
        if sims[i] < min_sim: break
        out.append(NODES.iloc[int(i)]["node_id"])
        if len(out) >= limit: break
    return out

def kg_search(q: str, limit: int = None) -> pd.DataFrame:
    limit = limit or KG["LIMIT"]
    rrf_k = KG["RRF_K"]
    scores = {}
    for rank, nid in enumerate(bm25_search(q, limit), 1):
        scores[nid] = scores.get(nid, 0.0) + 1.0/(rrf_k + rank)
    for rank, nid in enumerate(vec_search(q, limit, KG["VEC_MIN_SIM"]), 1):
        scores[nid] = scores.get(nid, 0.0) + 1.0/(rrf_k + rank)
    if not scores:
        return NODES.iloc[0:0].assign(rrf=[])
    ranked = sorted(scores.items(), key=lambda kv: -kv[1])
    idx = {r.node_id: r for r in NODES.itertuples()}
    rows = []
    for nid, sc in ranked[:limit]:
        r = idx.get(nid)
        if r is None: continue
        rows.append({"node_id": nid, "kind": r.kind, "label": r.label,
                     "importance": r.importance, "rrf": round(sc, 6), "text": r.text})
    return pd.DataFrame(rows)

kg_search("process mining conformance", limit=8)[["node_id","kind","rrf","label"]]


## 7) Граф: рёбра relation / in_wiki + подграф по глубине (BFS)

Как в панели: строим смежность, а при поиске раскрываем окрестность
найденных узлов на заданную **глубину связей** (ползунок в UI).

In [ ]:
import collections

def load_edges() -> pd.DataFrame:
    edges = []
    if table_exists(WORK_DB, "knowledge_relations"):
        rel = read_table(WORK_DB, "knowledge_relations")
        sc = _first_col(rel, "source_id", "from_id", "src_id")
        tc = _first_col(rel, "target_id", "to_id", "dst_id")
        lc = _first_col(rel, "relation_type", "label", "type")
        scorec = _first_col(rel, "relevance_score", "score")
        if sc and tc:
            for _, r in rel.iterrows():
                edges.append({"from": f"k{r[sc]}", "to": f"k{r[tc]}",
                              "label": str(r.get(lc, "") or "") if lc else "",
                              "kind": "relation",
                              "score": float(r.get(scorec, 0) or 0) if scorec else 0.0})
    # wiki-связи: wiki_pages.source_ids -> knowledge
    if table_exists(WORK_DB, "wiki_pages"):
        wk = read_table(WORK_DB, "wiki_pages")
        widc = _first_col(wk, "id"); srcc = _first_col(wk, "source_ids")
        if srcc:
            for _, r in wk.iterrows():
                raw = r.get(srcc, "")
                try:
                    ids = json.loads(raw) if isinstance(raw, str) and raw.strip().startswith("[") else                           [x for x in re.split(r"[,;\s]+", str(raw)) if x]
                except Exception:
                    ids = []
                for sid in ids:
                    edges.append({"from": f"k{sid}", "to": f"w{r[widc]}",
                                  "label": "in_wiki", "kind": "wiki_link", "score": 1.0})
    node_ids = set(NODES["node_id"])
    edges = [e for e in edges if e["from"] in node_ids and e["to"] in node_ids]
    return pd.DataFrame(edges)

EDGES = load_edges()
print("Рёбер:", len(EDGES),
      "| relation:", (EDGES.kind=="relation").sum() if len(EDGES) else 0,
      "| wiki_link:", (EDGES.kind=="wiki_link").sum() if len(EDGES) else 0)

ADJ = collections.defaultdict(set)
for e in EDGES.itertuples():
    ADJ[e._1 if hasattr(e,"_1") else e.__getattribute__("from")].add(e.to)
    ADJ[e.to].add(e._1 if hasattr(e,"_1") else e.__getattribute__("from"))
# надёжнее пересобрать по имени колонки:
ADJ = collections.defaultdict(set)
for _, e in EDGES.iterrows():
    ADJ[e["from"]].add(e["to"]); ADJ[e["to"]].add(e["from"])

def expand(seed_ids, depth: int):
    seen = set(seed_ids); frontier = set(seed_ids)
    for _ in range(max(0, int(depth))):
        nxt = set()
        for n in frontier:
            nxt |= ADJ.get(n, set())
        nxt -= seen; seen |= nxt; frontier = nxt
        if not frontier: break
    return seen

DEGREE = {nid: len(nei) for nid, nei in ADJ.items()}
print("Пример степеней (топ-5 хабов):",
      sorted(DEGREE.items(), key=lambda kv:-kv[1])[:5])


## 8) Панель визуала графа (vis-network) — в стиле дашборда

Рендерим подграф в HTML той же палитрой, что веб-панель:
knowledge — оранжево-мятные `dot`, wiki — сиреневые `star`, хабы крупнее,
рёбра relation — оранжевые, in_wiki — сиреневые. Клик по узлу открывает **карточку**.

In [ ]:
from IPython.display import HTML, display

PALETTE = dict(
    ink="#1E1A16", paper="#FAF3E7", orange="#FF5A1F",
    mint="#CDE8D5", lav="#E3D5F5", wikiEdge="#7b6f8f",
)

def _vis_nodes(node_ids, highlight=None):
    highlight = highlight or set()
    idx = {r.node_id: r for r in NODES.itertuples()}
    out = []
    for nid in node_ids:
        r = idx.get(nid)
        if r is None: continue
        is_wiki = (r.kind == "wiki")
        deg = DEGREE.get(nid, 0)
        size = 22 if deg >= KG["HUB_DEGREE"] else 12
        base_bg = PALETTE["lav"] if is_wiki else PALETTE["mint"]
        bg = PALETTE["orange"] if nid in highlight else base_bg
        out.append({
            "id": nid, "label": r.label, "shape": "star" if is_wiki else "dot",
            "size": size, "color": {"background": bg, "border": PALETTE["ink"]},
            "borderWidth": 4 if nid in highlight else 2,
            "font": {"face": "monospace", "size": 11, "color": PALETTE["ink"]},
            "title": (getattr(r, "text", "") or "")[:200],
        })
    return out

def _vis_edges(node_ids):
    s = set(node_ids); out = []
    for i, e in EDGES.iterrows():
        if e["from"] in s and e["to"] in s:
            col = PALETTE["wikiEdge"] if e["kind"] == "wiki_link" else PALETTE["orange"]
            out.append({"id": f"e{i}", "from": e["from"], "to": e["to"],
                        "label": e.get("label",""), "color": {"color": col},
                        "font": {"align":"middle","size":9,"color":"#3a342d"}, "width":1.4})
    return out

def render_graph(node_ids, highlight=None, height="560px"):
    nodes = _vis_nodes(node_ids, highlight)
    edges = _vis_edges(node_ids)
    payload = json.dumps({"nodes": nodes, "edges": edges})
    html = f"""
    <div style="border:2px solid {PALETTE['ink']};background:{PALETTE['paper']};padding:0;">
      <div id="kgnet" style="height:{height};"></div>
    </div>
    <script type="text/javascript" src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
    <script type="text/javascript">
    (function(){{
      var data = {payload};
      var container = document.getElementById('kgnet');
      var options = {{
        interaction:{{hover:true, navigationButtons:true, zoomView:true, dragView:true}},
        physics:{{enabled:true, stabilization:{{iterations:200}},
                 barnesHut:{{gravitationalConstant:-15000, springLength:210, springConstant:0.025, damping:0.35}}}},
        nodes:{{borderWidth:2}}, edges:{{width:1.2}}
      }};
      var net = new vis.Network(container, data, options);
      net.once('stabilizationIterationsDone', function(){{ net.setOptions({{physics:{{enabled:false}}}}); }});
      net.on('click', function(p){{ if(p.nodes && p.nodes[0]) window._kgSelected = p.nodes[0]; }});
    }})();
    </script>"""
    display(HTML(html))

# демонстрация: подграф вокруг первых узлов
_seed = list(NODES["node_id"].head(20))
render_graph(expand(_seed, KG["DEPTH_DEFAULT"]), highlight=set(_seed))


## 9) Карточка узла — как модалка `/api/graph/card`

Заголовок, meta (id/status/importance), тело (текст knowledge / wiki-md),
и, если есть, источник.

In [ ]:
def show_card(node_id: str):
    r = NODES[NODES.node_id == node_id]
    if r.empty:
        print("Нет такого узла:", node_id); return
    r = r.iloc[0]
    kind = "WIKI PAGE" if r.kind == "wiki" else "KNOWLEDGE"
    title = getattr(r, "title", None) or (r.text[:80] + ("…" if len(r.text) > 80 else ""))
    body = r.text.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    html = f"""
    <div style="border:2px solid {PALETTE['ink']};box-shadow:8px 8px 0 {PALETTE['orange']};
                background:{PALETTE['paper']};padding:18px 20px;max-width:760px;font-family:system-ui">
      <div style="font-family:monospace;font-size:11px;letter-spacing:.08em;color:#6b6055">{kind}</div>
      <h2 style="margin:6px 0 4px 0">{title}</h2>
      <div style="font-family:monospace;font-size:11px;color:#9a8f82">
        ID: {r.node_id} · status: {r.status or '-'} · importance: {r.importance:.2f}</div>
      <div style="margin-top:12px;line-height:1.55;white-space:pre-wrap">{body}</div>
    </div>"""
    display(HTML(html))

show_card(NODES.iloc[0]["node_id"])


## 10) Интерактивная панель поиска — поля ввода, кнопки, глубина, LLM

Полный аналог верхней части KG-панели:
- текстовое поле запроса + кнопка «Искать»;
- ползунок **глубины связей** (0–5);
- флажок **LLM-саммари**;
- вывод: перерисованный подграф с подсветкой найденного + список карточек.

In [ ]:
import ipywidgets as widgets

def llm_summarize(query: str, contexts: List[str]) -> str:
    if not LLM_CFG["api_key"] or not LLM_CFG["base_url"]:
        return "LLM-саммари выключено (задайте PM_LLM_BASE_URL и PM_LLM_API_KEY)."
    import urllib.request
    ctx = "\n\n".join(f"[{i+1}] {c[:800]}" for i, c in enumerate(contexts[:KG['MAX_CTX']]))
    prompt = (f"Запрос: {query}\n\nКонтекст (узлы графа знаний):\n{ctx}\n\n"
              "Сделай краткое саммари по теме запроса на основе контекста, "
              "ссылайся на источники вида [n].")
    body = json.dumps({"model": LLM_CFG["model"],
                       "messages":[{"role":"user","content":prompt}],
                       "max_tokens":700, "temperature":0.2}).encode()
    req = urllib.request.Request(LLM_CFG["base_url"].rstrip("/") + "/chat/completions",
                                 data=body, method="POST",
                                 headers={"Authorization": f"Bearer {LLM_CFG['api_key']}",
                                          "Content-Type":"application/json"})
    try:
        with urllib.request.urlopen(req, timeout=90) as resp:
            data = json.loads(resp.read())
        return data["choices"][0]["message"]["content"]
    except Exception as e:
        return f"LLM ошибка: {type(e).__name__}: {e}"

q_box   = widgets.Text(value="process mining", description="Запрос:",
                       layout=widgets.Layout(width="60%"))
depth   = widgets.IntSlider(value=KG["DEPTH_DEFAULT"], min=0, max=5, step=1,
                            description="Глубина:")
llm_chk = widgets.Checkbox(value=False, description="LLM-саммари")
go_btn  = widgets.Button(description="Искать", button_style="warning",
                         icon="search")
out     = widgets.Output()

def on_go(_):
    with out:
        out.clear_output()
        q = q_box.value.strip()
        if not q:
            print("Пустой запрос"); return
        found = kg_search(q, limit=KG["LIMIT"])
        if found.empty:
            print("Ничего не найдено"); return
        seeds = list(found["node_id"])
        sub = expand(seeds, depth.value)
        print(f"Найдено {len(found)} · подграф {len(sub)} узлов · depth {depth.value}")
        render_graph(sub, highlight=set(seeds))
        # карточки топ-5
        for nid in seeds[:5]:
            show_card(nid)
        if llm_chk.value:
            summ = llm_summarize(q, found["text"].tolist())
            display(HTML(f"<div style='border-left:4px solid {PALETTE['orange']};"
                         f"padding:8px 12px;margin-top:10px;white-space:pre-wrap'>"
                         f"<b>LLM-саммари</b><br>{summ}</div>"))

go_btn.on_click(on_go)
display(widgets.HBox([q_box, go_btn]))
display(widgets.HBox([depth, llm_chk]))
display(out)


## 11) Диагностика

Быстрая проверка индексов и параметров.

In [ ]:
print("Источник       :", SOURCE_MODE, "->", WORK_DB)
print("Узлов          :", len(NODES))
print("Рёбер          :", len(EDGES))
print("VECTOR_DIM     :", VECTOR_DIM, "(модель:", EMBED_MODEL_NAME, ")")
print("Матрица EMB    :", EMB.shape)
print("KG параметры   :", KG)
print("LLM            :", "on" if LLM_CFG["api_key"] else "off", "|", LLM_CFG["model"])
